In [ ]:
import pandas as pd
from sklearn.utils import shuffle
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler

In [ ]:
esa_21_22_fast_actions_parsed = pd.read_pickle('esa_21_22_fast_actions_parsed.pkl')
esa_22_23_fast_actions_parsed = pd.read_pickle('esa_22_23_fast_actions_parsed.pkl')
len(esa_22_23_fast_actions_parsed)

In [ ]:
euro_fast_actions_parsed = pd.read_pickle('euro_fast_actions_parsed.pkl')
fifa_fast_actions_parsed = pd.read_pickle('fifa_fast_actions_parsed.pkl')

In [ ]:
esa_22_23_core = pd.read_pickle('38_235.pkl')

In [ ]:
def get_test_set(df, df_core):
    dfs_test = []
    dfs_train = []
    teams = list(set(df['team'].values))
    test_matches = df_core[esa_22_23_core['match_week'] >= 18]
    train_matches = df_core[esa_22_23_core['match_week'] < 18]
    for team in teams:
        for match_id in test_matches:
            df_result = df[(df['team'] == team) & (df['match_id'] == match_id)]
            dfs_test.append(df_result)
        for match_id in train_matches:
            df_result = df[(df['team'] == team) & (df['match_id'] == match_id)]
            dfs_train.append(df_result)
    df_test = pd.concat(dfs_test)
    df_train = pd.concat(dfs_train)
    return esa_22_23_test, esa_22_23_train


esa_22_23_test, esa_22_23_train = get_test_set(esa_22_23_fast_actions_parsed)
print(len(esa_22_23_test))
print(len(esa_22_23_train))

In [ ]:
test_set = esa_22_23_test
train_set = pd.concat([esa_22_23_train, esa_21_22_fast_actions_parsed, euro_fast_actions_parsed, fifa_fast_actions_parsed])
print(len(test_set))
len(train_set)

In [ ]:
test_set['was_shot'] = [1 if (shots > 0) else 0 for shots in test_set['num_of_shot'].values]
negative_test = len(test_set[test_set['was_shot'] == 0])
positive_test = len(test_set[test_set['was_shot'] == 1])
print(negative_test)
positive_test


In [ ]:
train_set['was_shot'] = [1 if (shots > 0) else 0 for shots in train_set['num_of_shot'].values]
negative_train = len(train_set[train_set['was_shot'] == 0])
positive_train = len(train_set[train_set['was_shot'] == 1])
print(negative_train)
positive_train

In [ ]:
negative_total = negative_test + negative_train
positive_total = positive_test + positive_train
data_balance = negative_total / positive_total

In [ ]:
def get_sets_for_model(df_train, df_test):
    df_train = shuffle(df_train)
    df_test = shuffle(df_test)

    # scaler = preprocessing.MinMaxScaler(feature_range=(0, 1))

    # df_train_ids = df_train['id'].values  # type: ignore
    # df_test_ids = df_test['id'].values  # type: ignore
    # df_train_teams = df_train['team'].values  # type: ignore
    # df_test_teams = df_test['team'].values  # type: ignore
    # df_train_was_shot = df_train['was_shot'].values  # type: ignore
    # df_test_was_shot = df_test['was_shot'].values  # type: ignore
    # # df_train_good_ends = df_train['good_end'].values  # type: ignore
    # # df_test_good_ends = df_test['good_end'].values  # type: ignore

    # df_train = df_train.drop(columns=['id', 'team', 'path', 'start_area', 'end_area', 'was_shot']) # type: ignore
    # df_test = df_test.drop(columns=['id', 'team', 'path', 'start_area', 'end_area', 'was_shot']) # type: ignore

    df_train_columns = df_train.columns.sort_values()
    df_test_columns = df_test.columns.sort_values()

    # df_train = scaler.fit_transform(df_train)
    # df_test = scaler.fit_transform(df_test)

    df_train = pd.DataFrame(df_train, columns=df_train_columns)
    df_test = pd.DataFrame(df_test, columns=df_test_columns)

    # df_train['id'] = df_train_ids
    # df_test['id'] = df_test_ids
    # df_train['team'] = df_train_teams
    # df_test['team'] = df_test_teams
    # df_train['was_shot'] = df_train_was_shot
    # df_test['was_shot'] = df_test_was_shot

    y_train = df_train['was_shot']
    X_train = df_train.drop(columns=['was_shot'])

    y_test = df_test['was_shot']
    X_test = df_test.drop(columns=['was_shot'])

    return X_train, X_test, y_train, y_test

In [ ]:
X_train, X_test, y_train, y_test = get_sets_for_model(train_set, test_set)

In [ ]:
from xgboost import XGBClassifier

def train_model(X_train, X_test, y_train):
    model = XGBClassifier(objective='binary:logistic', scale_pos_weight=data_balance)

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    model.fit(input_X_train, y_train)

    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    XGB_predictions = model.predict(input_X_test)

    return model, XGB_predictions, input_X_test

model, pred, input_test = train_model(X_train, X_test, y_train)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

def evaluate_model(y_test, pred):
    print('Confusion matrix')
    print(confusion_matrix(y_test, pred))
    print('Classification report')
    print(classification_report(y_test, pred))
    print('Accuracy= ', accuracy_score(y_test, pred))

evaluate_model(y_test, pred)

In [ ]:
from datetime import datetime

# datetime object containing current date and time
now = datetime.now()
 
print("now =", now)

# dd/mm/YY H:M:S
dt_string = now.strftime("%d_%m_%Y_%H-%M-%S")
print("date and time =", dt_string)

In [ ]:
def feature_importances_raport(input_X_test, model, set_title):
    result = pd.DataFrame(columns=['name', 'value'])
    result['name'] = input_X_test.columns
    result['value'] = model.feature_importances_

    result = result.sort_values(by=['value'], ascending=False)

    result.to_csv(f'features_{set_title}.csv')
    print(result)
    return result

feature_importances = feature_importances_raport(input_test, model, f'fast_sequences_experiment_{dt_string}')

In [ ]:
def generate_scores(model, X_test, X_train, y_train, set_title):
    model = XGBClassifier(objective='binary:logistic')

    input_X_train = X_train.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    model.fit(input_X_train, y_train)
    
    input_X_test = X_test.drop(columns=['id', 'team', 'num_of_shot', 'xg', 'obv_for', 'obv_total', 'path', 'start_area', 'end_area'])
    model = model.predict_proba(input_X_test)

    X_test['score_positive'] = [m[1] for m in model]
    X_test['score_negative'] = [m[0] for m in model]

    X_test.to_csv(f'{set_title}_evaluated.csv')

    return X_test

df_with_scores = generate_scores(model, X_test, X_train, y_train, f'fast_sequences_experiment_{dt_string}')

In [ ]:
len(test_set[(test_set['team'] == 'Lech Poznan') & (test_set['num_of_shot'] > 0)])

In [ ]:
import statistics
import pandas as pd

def top_teams_by(df_with_scores, by, dt_string):
    teams = list(set(df_with_scores['team'].values))
    df_team_scores = pd.DataFrame()
    df_scores = []
    df_teams = []
    df_obv = []
    df_xg = []
    df_sequences = []
    df_shots = []
    for team in teams:
        df_with_scores_team = df_with_scores[df_with_scores['team'] == team]
        if by == 'median':
            score = statistics.median(df_with_scores_team['score_positive'].values)
            obv = statistics.median(df_with_scores_team['obv_for'].values)
            xg = statistics.median(df_with_scores_team['xg'].values)
        elif by == 'mean':
            score = statistics.mean(df_with_scores[df_with_scores['team'] == team]['score_positive'].values)
            obv = statistics.mean(df_with_scores[df_with_scores['team'] == team]['obv_for'].values)
            xg = statistics.mean(df_with_scores[df_with_scores['team'] == team]['xg'].values)
        else:
            score = sum(df_with_scores[df_with_scores['team'] == team]['score_positive'])
            obv = sum(df_with_scores[df_with_scores['team'] == team]['obv_for'])
            xg = sum(df_with_scores[df_with_scores['team'] == team]['xg'])
        sequences = len(df_with_scores_team)
        shots = len(df_with_scores_team[df_with_scores_team['num_of_shot'] > 0])
        df_teams.append(team)
        df_obv.append(obv)
        df_xg.append(xg)
        df_scores.append(score)
        df_sequences.append(sequences)
        df_shots.append(shots)
    df_team_scores['team'] = df_teams
    df_team_scores['score'] = df_scores
    df_team_scores['obv'] = df_obv
    df_team_scores['xg'] = df_xg
    df_team_scores['sequences'] = df_sequences
    df_team_scores['shots'] = df_shots
    result = df_team_scores.sort_values(by=['score'], ascending=False)
    result.to_csv(f'fast_sequences_top_teams_{by}_{dt_string}.csv')
    return result

In [ ]:
top_teams_by(df_with_scores, 'sum', dt_string)

In [ ]:
top_teams_by(df_with_scores, 'mean', dt_string)

In [ ]:
top_teams_by(df_with_scores, 'median', dt_string)

In [ ]:
def show_correlation(df_with_scores, dt_string):
    corr_with_xg = df_with_scores['score_positive'].corr(df_with_scores['xg'])
    corr_with_obv = df_with_scores['score_positive'].corr(df_with_scores['obv_for'])
    corr_with_obv_total = df_with_scores['score_positive'].corr(df_with_scores['obv_total'])
    print(f'Correlation with xg: {corr_with_xg:.3f}')
    print(f'Correlation with obv_for: {corr_with_obv:.3f}')
    print(f'Correlation with obv_total: {corr_with_obv_total:.3f}')
    result = pd.DataFrame()
    result['xg_corr'] = corr_with_xg
    result['obv_corr'] = corr_with_obv
    result['obv_total'] = corr_with_obv_total
    result.to_csv(f'fast_sequences_correlations_{dt_string}.csv')

In [ ]:
show_correlation(df_with_scores, dt_string)